# CS 696 · Defend an evidence-grounded result

Reproduce an event-QA score from frozen records, state its scope, and defend the assumptions behind a model-based what-if answer.

Module 07 supplies two virtual cart prefixes and four questions. You already know which answers refer to observations, which depend on a transition model and which future is unresolved. Now give the same materials to a second reader. Could they detect an answer that has the right word but gives too little evidence?

The candidate sheet in this fictional audit contains two deliberate faults. E says “yes” and cites only tick 2. F gives the conditional answer with its model and starting tick. U abstains. W says “yes” where the cart remained in green. The numerical results below describe those four constructed responses.

## Determine what the score actually measures

Give each question one vote and freeze the scoring contract before running it. Short-answer scoring compares only the value. Complete-response scoring also checks the answer type, canonical evidence ticks and model ID. For these selected questions Module 07 supplies one accepted annotation per item. In a larger task, independent annotators might identify several sufficient evidence sets. Agree on how to accept alternatives before scoring them.

Sshort = Σᵢ 1[valueᵢ = reference valueᵢ] / n     Sfull = Σᵢ 1[all four fields match] / n

The indicator 1[⋅] equals one when its condition is true. Each response carries a status, value, tick tuple and optional model ID. Here n equals four. A full score needs both the answer and its recorded justification.

In [1]:
# E, F, U and W use the question order from the independent answer sheet.
gold = {'E': ('observed', 'yes', (1, 2), None),
        'F': ('conditional', 'green', (2,), 'advance-v1'),
        'U': ('unresolved', 'unknown', (), None),
        'W': ('observed', 'no', (1, 2), None)}
predicted = dict(gold)
predicted['E'] = ('observed', 'yes', (2,), None)
predicted['W'] = ('observed', 'yes', (1, 2), None)
short = [q for q in gold if predicted[q][1] == gold[q][1]]
full = [q for q in gold if predicted[q] == gold[q]]
assert short == ['E', 'F', 'U'] and full == ['F', 'U']
print('Correct short answer:', short, 'Complete match:', full)


Correct short answer: ['E', 'F', 'U'] Complete match: ['F', 'U']


**Discuss** · Why do the candidates for E and W require different feedback?

E needs the previous tick to support the word “entered.” Its answer value is right. W cites the correct pair of ticks but misreads two green observations as an entry, so its value is wrong. Merely adding citations would leave W incorrect.

## Make a second reader’s input verifiable

Export one row per question, keeping unanswered future values distinct from missing spreadsheet cells. The literal word unknown is U’s deliberate answer. A blank ticks field denotes the empty evidence list. Record the frozen file’s SHA-256 digest separately. A reader who edits E’s evidence after seeing the errors will get a different digest. Preserve the original predictions and score any repairs as a separate version.

h = SHA256(B)     input verified when h = hrecorded

B is the entire UTF-8 byte sequence of the frozen CSV, including its header and final newline. A digest checks byte identity. It does not establish that the reference evidence is scientifically appropriate. The second reader should separately compare the reference with the raw prefixes.

In [2]:
from hashlib import sha256

rows = b"""id,kind,answer,ticks,model,p_kind,p_answer,p_ticks,p_model
E,observed,yes,1|2,,observed,yes,2,
F,conditional,green,2,advance-v1,conditional,green,2,advance-v1
U,unresolved,unknown,,,unresolved,unknown,,
W,observed,no,1|2,,observed,yes,1|2,
"""
recorded = 'e5f03e7ae4fdb97743a699af955852356c3d2f95bd5d1733e95a85987569d22d'
assert sha256(rows).hexdigest() == recorded
edited = rows.replace(b'observed,yes,2,', b'observed,yes,1|2,', 1)
assert rows != edited and sha256(edited).hexdigest() != recorded
print('Original input verified. Candidate edit changes digest.')


Original input verified. Candidate edit changes digest.


**Discuss** · If the input digest matches but another reader awards E a full point, what should you compare next?

Check the evaluator revision and its field list. An evaluator comparing only type and value would silently omit the evidence requirement. Have the reader show E’s reference and candidate ticks next to the predicate used to award the point.

## Defend the causal interpretation of a what-if

The mentor asks whether forcing ADVANCE causes the next zone to be green. In the specified toy transition model, compare WAIT and ADVANCE from precisely the same blue state. A second plausible model permits a side route during ADVANCE. Both models can share the entire observed prefix. A claim about what the virtual cart would actually have done additionally requires a correct transition rule, the same starting conditions and background route setting in both branches, and a well-defined action intervention. The prefix alone cannot establish those assumptions.

Δm(s) = 1[zone(Tm(s, ADVANCE)) = green] − 1[zone(Tm(s, WAIT)) = green]

Δm is a contrast between two modeled actions for the same starting state s and fixed model m. The indicator is one when the resulting zone is green. This difference is a statement about these modeled branches, without an estimated causal effect in an external environment.

In [3]:
def transition(state, action, route):
    tick, zone = state
    forward = {'amber': 'blue', 'blue': 'green', 'green': 'amber'}
    if action not in ('WAIT', 'ADVANCE') or route not in ('main', 'side'):
        raise ValueError('Unknown action or route')
    if action == 'WAIT':
        return tick+1, zone
    next_zone = 'amber' if (zone, route) == ('blue', 'side') else forward[zone]
    return tick+1, next_zone

start = (2, 'blue')
contrasts = {}
for route in ('main', 'side'):
    policy_branch = transition(start, 'WAIT', route)
    forced_branch = transition(start, 'ADVANCE', route)
    contrasts[route] = int(forced_branch[1] == 'green') - int(policy_branch[1] == 'green')
assert contrasts == {'main': 1, 'side': 0}
print('Action contrast with route fixed within each pair:', contrasts)


Action contrast with route fixed within each pair: {'main': 1, 'side': 0}


**Discuss** · How would you transfer this distinction to a document QA system and to a simulator generated from a process description?

For a document question, cite the passage supporting an observation or rule, and mark assumptions needed for a hypothetical answer. For a generated simulator, freeze the initial state, rule version, and any random inputs before changing one action. Agreement with one historical outcome alone cannot validate its alternative-action branch.

## Independent task

Create a reproducibility handoff for a one-page memo. Verify the frozen input, run a versioned scorer in a fresh Python process and compare the result with the expected counts you can derive from E, F, U and W. Test that an unannounced edit is rejected. In the memo give the error mechanism, the case IDs supporting it and the boundary of the claim.

In [4]:
from pathlib import Path
from tempfile import TemporaryDirectory
from hashlib import sha256

cases = b"""id,kind,answer,ticks,model,p_kind,p_answer,p_ticks,p_model
E,observed,yes,1|2,,observed,yes,2,
F,conditional,green,2,advance-v1,conditional,green,2,advance-v1
U,unresolved,unknown,,,unresolved,unknown,,
W,observed,no,1|2,,observed,yes,1|2,
"""
with TemporaryDirectory() as folder:
    path = Path(folder, 'cases.csv')
    path.write_bytes(cases)
    assert sha256(path.read_bytes()).hexdigest() == 'e5f03e7ae4fdb97743a699af955852356c3d2f95bd5d1733e95a85987569d22d'
    print('Frozen input verified:', path.name)
# Add the evaluator, record its digest and rerun it independently of this process.


Frozen input verified: cases.csv


## Compare your solution

Keep the expected counts in the manifest and compute the actual counts in the scorer. Compare the expected and actual results after a separate interpreter has read the CSV. Use the raw prefixes to check the scientific meaning of E and W.

In [5]:
from hashlib import sha256
from pathlib import Path
from tempfile import TemporaryDirectory
import json
import subprocess
import sys

cases = b"""id,kind,answer,ticks,model,p_kind,p_answer,p_ticks,p_model
E,observed,yes,1|2,,observed,yes,2,
F,conditional,green,2,advance-v1,conditional,green,2,advance-v1
U,unresolved,unknown,,,unresolved,unknown,,
W,observed,no,1|2,,observed,yes,1|2,
"""
script = """import csv
from pathlib import Path

with Path('cases.csv').open(encoding='utf-8', newline='') as stream:
    rows = list(csv.DictReader(stream))
assert [r['id'] for r in rows] == ['E', 'F', 'U', 'W']
fields = ('kind', 'answer', 'ticks', 'model')
short = full = 0
for row in rows:
    short += row['answer'] == row['p_answer']
    full += all(row[key] == row['p_'+key] for key in fields)
print(f'{len(rows)} cases, {short} short answers, {full} complete responses')
"""
with TemporaryDirectory() as folder:
    root = Path(folder)
    (root/'cases.csv').write_bytes(cases)
    (root/'score.py').write_bytes(script.encode('utf-8'))
    manifest = {'runtime': 'Python 3, standard library', 'command': ['score.py'],
                'expected': '4 cases, 3 short answers, 2 complete responses',
                'sha256': {'cases.csv': 'e5f03e7ae4fdb97743a699af955852356c3d2f95bd5d1733e95a85987569d22d',
                           'score.py': '1e7c6e5e0edb16f04d23a5647b972dc559554fa1ccec0ca0034dfafed9c1c7ae'}}
    (root/'manifest.json').write_text(json.dumps(manifest), encoding='utf-8')
    received = json.loads((root/'manifest.json').read_text(encoding='utf-8'))
    def verified():
        return all(sha256((root/name).read_bytes()).hexdigest() == expected
                   for name, expected in received['sha256'].items())
    assert verified()
    run = subprocess.run([sys.executable, *received['command']], cwd=root,
                         check=True, text=True, capture_output=True)
    assert run.stdout.strip() == received['expected']
    print(run.stdout.strip())
    (root/'cases.csv').write_bytes(cases.replace(b'observed,yes,2,', b'observed,yes,1|2,', 1))
    assert not verified()
    print('Edited input rejected')


4 cases, 3 short answers, 2 complete responses
Edited input rejected


On this frozen four-item teaching fixture, the candidate matched three short answers and two complete responses. E accounts for the difference because its evidence omits the previous tick. W fails both scores. These counts characterize this candidate sheet and these questions. They do not estimate accuracy on new episodes.

## Mentor review

Provide a one-page memo, a handoff bundle with visible prefixes, human-readable questions, reference and candidate answers, transition rule, manifest and scorer, and a brief oral defense. Have another reader run the command and note their interpreter version, output and any clarification required.

- A reader reconstructs E and W from the raw prefixes, checks both digests, and independently reproduces the 3/4 and 2/4 scores.
- The claim names the four-case fixture and identifies the single evidence omission that separates the scores.
- In the defense, distinguish observation, prediction under a policy, and the modeled effect of changing an action. Explain one assumption that alters the last result.
- Which piece of evidence would make U answerable as a conditional prediction? What further observation would establish the actual next zone?
- Would the next study learn more from new episodes or new questions about the same episodes? Choose a unit of generalization and name the test set before developing the next rule.

A concrete next study asks whether requiring both adjacent observations reduces unsupported entry claims on unseen runs with occasional missing ticks. Freeze the event definition and both answer policies. Reserve whole episode groups, keep suffixes hidden, and have independent readers label answerability and acceptable evidence before scoring. Report unsupported claims together with erroneous abstentions. Return to the mentoring meeting with the blinded case inventory and the first disagreement to adjudicate.